# 🛡️ Phase 2: Building the Privacy Shield (Defense Implementation)

**Hardware:** CPU (No GPU needed yet)
**Input:** `pii_simulation_data.csv` (Created in Phase 1)

## 🎯 Objective
In this notebook, we will:
1.  **Build the Defense:** Initialize Microsoft Presidio (Analyzer & Anonymizer).
2.  **Test the Logic:** Run our "Leaking" and "Safe" data through the defense.
3.  **Evaluate:** Calculate the "Success Rate" (Did we catch all secrets?) and "False Positive Rate" (Did we accidentally block safe drugs?).

In [1]:
# Install Microsoft Presidio libraries
!pip install presidio-analyzer presidio-anonymizer spacy
# Download the English language model for text analysis
!python -m spacy download en_core_web_lg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.7/128.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 1. Load the Simulated Data
We mount Google Drive and load the CSV file we created in the previous phase.

In [3]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

# Path to your project data
project_path = '/content/drive/My Drive/MedGemma_Security_Project/Data'
file_path = os.path.join(project_path, 'pii_simulation_data.csv')

# Load Data
df = pd.read_csv(file_path)
print(f"✅ Loaded {len(df)} samples from {file_path}")
display(df.head())

Mounted at /content/drive
✅ Loaded 400 samples from /content/drive/My Drive/MedGemma_Security_Project/Data/pii_simulation_data.csv


,text,label,entity_type
0,Scan revealed damage to the Left Ventricle.,SAFE,NaN
1,The subject's social security number is 304-31...,LEAK,ssn
2,Send the diagnosis to +1-358-479-1733x5748 imm...,LEAK,phone
3,Please contact 515.542.7537x36202 regarding th...,LEAK,phone
4,Please contact 849-71-2637 regarding their lab...,LEAK,ssn


## 2. The Defense Logic
We initialize the **Analyzer** (The Detective) and the **Anonymizer** (The Censor).
We verify that it detects specific HIPAA categories: `PERSON`, `PHONE_NUMBER`, `EMAIL_ADDRESS`, `US_SSN`.

In [4]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# Initialize the engines
# 'en_core_web_lg' is a large English model that understands context better
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def privacy_shield(text):
    """
    Input: A string (potentially containing secrets).
    Output: A sanitized string with secrets replaced by <REDACTED>.
    """
    # 1. Analyze (Find the secrets)
    # We look for specific PII entities
    results = analyzer.analyze(text=text,
                               entities=["PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS", "US_SSN"],
                               language='en')

    # 2. Anonymize (Hide the secrets)
    # We define how we want to hide them. Let's use <REDACTED>
    anonymized_result = anonymizer.anonymize(text=text, analyzer_results=results)

    return anonymized_result.text

# --- Quick Test ---
test_msg = "Patient John Doe has a phone number 555-0199."
print(f"Original: {test_msg}")
print(f"Filtered: {privacy_shield(test_msg)}")

Original: Patient John Doe has a phone number 555-0199.
Filtered: Patient <PERSON> has a phone number 555-0199.


## 3. Evaluation Loop
We will now apply the `privacy_shield` to every row in our dataset.
*   If the input was **LEAK**, the output should contain `<REDACTED>` (Success).
*   If the input was **SAFE**, the output should be identical to the input (Success).

In [5]:
# Apply the shield to the 'text' column
df['cleaned_text'] = df['text'].apply(privacy_shield)

# Define Logic for Success
def check_success(row):
    # Case 1: It was a LEAK sample
    if row['label'] == 'LEAK':
        # Success if we see the placeholder < (from <REDACTED> or <PERSON>)
        if "<" in row['cleaned_text']:
            return "✅ Blocked"
        else:
            return "❌ Missed (Danger)"

    # Case 2: It was a SAFE sample
    elif row['label'] == 'SAFE':
        # Success if text didn't change
        if row['cleaned_text'] == row['text']:
            return "✅ Passed"
        else:
            return "⚠️ Over-censored (False Positive)"

df['result'] = df.apply(check_success, axis=1)

# Show results
display(df[['label', 'text', 'cleaned_text', 'result']].head(10))

,label,text,cleaned_text,result
0,SAFE,Scan revealed damage to the Left Ventricle.,Scan revealed damage to the Left Ventricle.,✅ Passed
1,LEAK,The subject's social security number is 304-31...,The subject's social security number is <US_SSN>.,✅ Blocked
2,LEAK,Send the diagnosis to +1-358-479-1733x5748 imm...,Send the diagnosis to +1-358-479-1733x5748 imm...,❌ Missed (Danger)
3,LEAK,Please contact 515.542.7537x36202 regarding th...,Please contact <PHONE_NUMBER> regarding their ...,✅ Blocked
4,LEAK,Please contact 849-71-2637 regarding their lab...,Please contact <US_SSN> regarding their lab re...,✅ Blocked
5,LEAK,Patient identifier: nancy77@example.com.,Patient identifier: <EMAIL_ADDRESS>.,✅ Blocked
6,LEAK,The subject's social security number is Anna F...,The subject's social security number is <PERSON>.,✅ Blocked
7,LEAK,Patient identifier: Andrew George.,Patient identifier: <PERSON>.,✅ Blocked
8,LEAK,The subject's social security number is 434.29...,The subject's social security number is <PHONE...,✅ Blocked
9,LEAK,Patient identifier: sullivananna@example.com.,Patient identifier: <EMAIL_ADDRESS>.,✅ Blocked


## 4. Final Metrics
We calculate our scores to report in the paper.

In [6]:
# Calculate Block Rate for Leaks (We want 100%)
leak_df = df[df['label'] == 'LEAK']
successful_blocks = len(leak_df[leak_df['result'] == "✅ Blocked"])
block_rate = (successful_blocks / len(leak_df)) * 100

# Calculate False Positive Rate for Safe Data (We want 0%)
safe_df = df[df['label'] == 'SAFE']
false_positives = len(safe_df[safe_df['result'] == "⚠️ Over-censored (False Positive)"])
fp_rate = (false_positives / len(safe_df)) * 100

print(f"🛡️ PRIVACY SHIELD RESULTS:")
print(f"--------------------------")
print(f"Safety Score (Recall):       {block_rate:.2f}% (Goal: 100%)")
print(f"Utility Score (False Pos):   {fp_rate:.2f}%   (Goal: 0%)")

# Save results for the final report
save_path = os.path.join(project_path, 'defense_results_cpu.csv')
df.to_csv(save_path, index=False)

🛡️ PRIVACY SHIELD RESULTS:
--------------------------
Safety Score (Recall):       93.00% (Goal: 100%)
Utility Score (False Pos):   0.00%   (Goal: 0%)
